In [ ]:
import re
import glob
import os
import pandas as pd

In [ ]:
folder_path = 'dataset'
file_pattern = os.path.join(folder_path, "*.txt")
file_list = glob.glob(file_pattern)

all_data = []

# วนลูปอ่านทีละไฟล์
for file_path in file_list:
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
        lines = file.readlines()

    equipment = ""
    meas_point = ""
    datetime_val = ""
    start_reading = False

    for line in lines:
        # สกัดข้อมูล Header
        if "Equipment:" in line:
            equipment = line.split("Equipment:")[1].strip()
        elif "Meas. Point:" in line:
            meas_point = line.split("Meas. Point:")[1].strip()
        elif "Date/Time:" in line:
            match = re.search(r'Date/Time:\s+(.*?)\s+Amplitude:', line)
            if match:
                datetime_val = match.group(1).strip()
        
        # เริ่มอ่านข้อมูลตัวเลขหลังเส้นขีด ---------
        if "---------" in line:
            start_reading = True
            continue
        
        if start_reading and line.strip():
            parts = line.split()
            # วนลูปอ่านข้อมูล 4 คู่ (Time, Amplitude) ในหนึ่งบรรทัด
            for i in range(0, len(parts), 2):
                if i + 1 < len(parts):
                    try:
                        time_ms = float(parts[i])
                        amplitude = float(parts[i+1])
                        # เก็บข้อมูลลง List
                        all_data.append([equipment, meas_point, datetime_val, time_ms, amplitude])
                    except ValueError:
                        continue

# สร้าง DataFrame รวมจากทุกไฟล์
ml_dataset = pd.DataFrame(all_data, columns=['Equipment', 'Meas. Point', 'Datetime', 'Time (mS)', 'Amplitude'])
# ml_dataset.to_csv('ml_dataset.csv', index=False)